In [8]:
import sqlite3
import json

In [ ]:
DB_PATH = "data.nosync/recipes.db"
RECPIPES_TABLE = "recipes"
RECIPES_ING_PROCESSED = "recipes_ingredients_processed"
INGREDIENTS_TABLE = "ingredients"
BATCH_SIZE = 1000


con = sqlite3.connect(DB_PATH)
tables = con.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()

if (INGREDIENTS_TABLE,) not in tables:
  print(f"Creating table {INGREDIENTS_TABLE}...")
  con.execute(
    f"""
      CREATE TABLE {INGREDIENTS_TABLE} (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        recipe_url TEXT NOT NULL,
        original_desc TEXT NOT NULL,
        ingredient_eng TEXT,
        amount TEXT,
        unit TEXT
      )
    """
  )
  con.commit()
  
cols_in_recipes = con.execute(f"PRAGMA table_info({RECPIPES_TABLE});").fetchall()
if RECIPES_ING_PROCESSED not in [col[1] for col in cols_in_recipes]:
  print(f"Adding column {RECIPES_ING_PROCESSED} to table {RECPIPES_TABLE}...")
  con.execute(f"ALTER TABLE {RECPIPES_TABLE} ADD COLUMN {RECIPES_ING_PROCESSED} INTEGER DEFAULT 0;")
  con.commit()

def fetch_batch(batch_size):
  return con.execute(
    f"""
    SELECT url, ingredients
    FROM recipes
    where {RECIPES_ING_PROCESSED} = 0
    LIMIT {batch_size}
    """
  ).fetchall()
  
  
def process_batch(batch):
  for rowid, ingredients in batch:
    ingredients_list = ingredients_list = json.loads(ingredients)
    
    updates = [(rowid, ingredient) for ingredient in ingredients_list]
    con.executemany(
      f"""
      INSERT INTO {INGREDIENTS_TABLE} (recipe_url, original_desc)
      VALUES (?, ?)
      """,
      updates
    )
    
    # update also the entry in the recipes 
    con.execute(
      f"""UPDATE {RECPIPES_TABLE} SET {RECIPES_ING_PROCESSED} = 1 WHERE url = ?""",
      (rowid,)
    )
    
    con.commit()
  
  
total_count = con.execute(f"SELECT COUNT(*) FROM {RECPIPES_TABLE} WHERE {RECIPES_ING_PROCESSED} = 0").fetchone()[0]
print(f"Total recipes to process: {total_count}")
total_batches = (total_count + BATCH_SIZE - 1) // BATCH_SIZE + 5
print(f"Total batches to process: {total_batches}")
  
for i in range(total_batches):
  print(f"Processing batch {i + 1}/{total_batches}...")
  batch = fetch_batch(BATCH_SIZE)
  if batch:
    process_batch(batch)

Creating table ingredients...
Adding column recipes_ingredients_processed to table recipes...
